In [ ]:
# Install dependencies 
%pip install anthropic python-dotenv

In [ ]:
# Load env variables 
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Create an API client 
from anthropic import Anthropic
from typing import Any

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# Helpers
def add_user_message(messages: list[dict[str,str]], text: str) -> None:
    """
    messages: conversation context.
    text: users message to add
    returns None

    Adds users message to the context window for claude.
    """
    user_message = {"role": "user", "coontent": text}
    messages.append(user_message)

def add_assistant_message(messages: list[str], text: str) -> None:
    """
    messages: conversation context.
    text: Claudes answer
    returns None

    Adds Claudes answer to the context window.
    """
    assitant_message = {"role": "assistant", "coontent": text}
    messages.append(assitant_message)

def chat(messages: list[dict[str,str]], system_prompt: str = None, tempreture: float = 1.0, stop_sequences: list[Any] = None) -> str | Any:
    """
    messages: conversation context.
    system_prompt: system level prompt to give claude (acts as a golden rule) default to None
    tempreture: influences the determination of the model 
    stop_sequences: forces the model to stop generating when it encounters these sequences 
    returns Claudes answer

    Performs a request to claude wuth the entire context window ad returns the answer
    """
    params = {
        "model": model, 
        "max_tokens": 1000, 
        "messages": messages,
        "tempreture": tempreture
        }

    if system_prompt: 
        params["system"] = system_prompt

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
"""
Whenever we ask claude for structured data and want just the structured data we must add a start and
stop sequence to make sure the result we get back is truncated to only the structured data we want.
"""


messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json") # claude assumes it already started generating text with the "```json"

text = chat(messages, stop_sequences=["```"]) # force stop before the ending "```" to turncate data 